# Semana 7 — LOINC y armonización de resultados de laboratorio

**Escenario (simulado, pero con las reglas de un proyecto real en Australia):** el hospital empieza a recibir un extracto diario del laboratorio de patología. El laboratorio no envía LOINC: envía **sus propios códigos** (`CHOL`, `GLU`, `A1CI`...), casi todo en unidades SI, pero un analizador antiguo todavía informa en mg/dL, algunos resultados llegan corregidos, otros vienen censurados (`<30`) y hay pacientes que no se pueden vincular. Al mismo tiempo, la historia clínica (HCE, aquí Synthea) ya tiene resultados con LOINC de EE. UU. en mg/dL. Tu trabajo: **mapear los códigos locales a LOINC, validarlos contra el servidor nacional, armonizar las unidades a SI con UCUM, unir ambas fuentes sin contar dos veces el mismo resultado y dejar todo trazable.**

**Productos y estándares reales:** LOINC (Regenstrief Institute), UCUM, National Clinical Terminology Service (NCTS), conjuntos de referencia SPIA del RCPA (*Standards for Pathology Informatics in Australia*), datos de prueba oficiales de HL7 Australia (`au-fhir-test-data`) y las ecuaciones de conversión de HbA1c publicadas por el NGSP.

**Al terminar tendrás:** `data/reference/loinc_crosswalk.csv` (tabla de equivalencias versionable), `data/reference/loinc_unmapped_queue.csv` (cola de revisión), `data/silver/lab_results.parquet` (resultados armonizados) y su manifiesto de linaje.

**Requisitos:** semanas 3 y 6. Credenciales del NCTS opcionales (Colab Secrets `NCTS_CLIENT_ID` y `NCTS_CLIENT_SECRET`, igual que en la semana 6); sin ellas, las partes en línea se omiten y el resto funciona igual.

**Importante:** el extracto del laboratorio lo **generas tú** en la Parte 0 a partir de Synthea. Es sintético y el laboratorio es ficticio; lo que es real son las reglas que vas a aplicar.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Librerías, carpetas y resultados de la HCE

**Explicación línea por línea**

- `import hashlib, json, re, subprocess, time`: huellas SHA-256 para el linaje, JSON para el manifiesto, expresiones regulares para leer resultados como `<30`, `subprocess` para clonar un repositorio con `git` y `time` para pausar entre llamadas al servidor.
- `import numpy as np` / `import pandas as pd`: cálculo numérico y tablas.
- `RAW_DIR`, `REF_DIR`, `REPORTS_DIR`, `EXT_DIR`: entrada cruda (el extracto del laboratorio), tablas de referencia, reportes y datos externos. `mkdir(parents=True, exist_ok=True)` las crea si faltan.
- `pd.read_parquet(..., columns=[...])`: lee de silver (semana 3) solo las columnas necesarias. **Por qué:** Parquet es columnar; pedir columnas reduce memoria y tiempo.

In [ ]:
import hashlib
import json
import re
import subprocess
import time

import numpy as np
import pandas as pd

SILVER_DIR = DATA_DIR / "silver"
RAW_DIR = DATA_DIR / "raw"
REF_DIR = DATA_DIR / "reference"
REPORTS_DIR = DATA_DIR / "reports"
EXT_DIR = DATA_DIR / "external"
for d in (RAW_DIR, REF_DIR, REPORTS_DIR, EXT_DIR):
    d.mkdir(parents=True, exist_ok=True)

obs = pd.read_parquet(SILVER_DIR / "observations.parquet",
                      columns=["obs_ts", "patient_id", "encounter_id", "loinc_code", "description", "value_num", "unit"])
print(f"{len(obs):,} observaciones en silver")

## Parte 0 — Simular el extracto del laboratorio de patología

### Paso 2 — Especificación del laboratorio ficticio

Un laboratorio real entrega un **catálogo de pruebas** (código local, nombre, tipo de muestra, unidad). Aquí lo defines tú, siguiendo las convenciones australianas: resultados en SI (mmol/L, µmol/L) y HbA1c informada **en dos unidades**, % (NGSP) y mmol/mol (IFCC).

**Explicación línea por línea**

- `rng = np.random.default_rng(20261122)`: generador aleatorio con **semilla fija**. Toda la simulación es determinista: si la repites, obtienes exactamente el mismo extracto. **En la práctica real** cualquier dato simulado para pruebas se genera con semilla, si no los tests no son reproducibles.
- `SPEC = {...}`: para cada LOINC de Synthea, la prueba equivalente del laboratorio: `(código local, nombre local, muestra, unidad, factor de conversión, decimales informados)`.
- Factores: colesterol (total, HDL, LDL) × 0,02586 y triglicéridos × 0,01129 son los que publica el NHANES (CDC) para pasar de mg/dL a mmol/L; glucosa × 0,0555 es el factor de las tablas de conversión SI del BMJ; creatinina × 88,4 pasa de mg/dL a µmol/L. **El simulador usa estos factores "de tabla" y el armonizador (Parte E) usará masas molares; que ambos caminos coincidan será una prueba independiente de tus conversiones.**
- `"Blood"` para glucosa y creatinina: en un laboratorio australiano real ambas suelen medirse en suero o plasma. Aquí se mantiene sangre total porque los valores vienen de códigos de Synthea en sangre (`2339-0`, `38483-4`). **Cambiar el tipo de muestra sin cambiar el valor sería un error semántico:** en LOINC la muestra es un eje que define el significado del código.
- `OLD_ANALYSER`: pruebas que el analizador antiguo informa en mg/dL.
- `src = obs[...]`: resultados de laboratorio de Synthea desde 2019 con valor numérico; serán "lo que el laboratorio envió".
- `.drop_duplicates(["patient_id", "loinc_code", "obs_ts"])`: Synthea a veces registra el mismo código dos veces en el mismo instante; un laboratorio informa una vez por muestra y prueba.
- `draws = ...drop_duplicates()`: cada combinación paciente + hora de toma es una **muestra** (un tubo), identificada por un número de accesión (`accession`), como en un sistema de información de laboratorio (LIS).
- `draws["old_analyser"] = rng.random(len(draws)) < 0.03`: el 3 % de las muestras pasa por el analizador antiguo.
- `pd.to_timedelta(rng.integers(2, 30, len(draws)), unit="h")`: la hora de informe llega entre 2 y 29 horas después de la toma.
- `ur`: cada paciente recibe un **UR number** (*Unit Record number*, el número de historia clínica de un hospital australiano) inventado con 7 dígitos. `rng.choice(9_000_000, size=..., replace=False)` garantiza que no se repitan.

In [ ]:
rng = np.random.default_rng(20261122)

SPEC = {
    "2093-3": ("CHOL", "Cholesterol", "Serum", "mmol/L", 0.02586, 1),
    "2085-9": ("HDL", "HDL Cholesterol", "Serum", "mmol/L", 0.02586, 2),
    "18262-6": ("LDL", "LDL Cholesterol", "Serum", "mmol/L", 0.02586, 1),
    "2571-8": ("TRIG", "Triglycerides", "Serum", "mmol/L", 0.01129, 1),
    "2339-0": ("GLU", "Glucose", "Blood", "mmol/L", 0.0555, 1),
    "38483-4": ("CREAT", "Creatinine", "Blood", "umol/L", 88.4, 0),
}
OLD_ANALYSER = {"CHOL", "HDL", "LDL", "TRIG", "GLU"}

src = (obs[obs["loinc_code"].isin(list(SPEC) + ["4548-4"]) & obs["value_num"].notna() & (obs["obs_ts"] >= "2019-01-01")]
       .drop_duplicates(["patient_id", "loinc_code", "obs_ts"])
       .sort_values(["patient_id", "obs_ts", "loinc_code"]).reset_index(drop=True))

draws = src[["patient_id", "obs_ts"]].drop_duplicates().reset_index(drop=True)
draws["accession"] = [f"SIM{t:%y}-{i:06d}" for i, t in enumerate(draws["obs_ts"], start=1)]
draws["old_analyser"] = rng.random(len(draws)) < 0.03
draws["reported_ts"] = draws["obs_ts"] + pd.to_timedelta(rng.integers(2, 30, len(draws)), unit="h")
src = src.merge(draws, on=["patient_id", "obs_ts"], how="left", validate="many_to_one")

patients = np.sort(src["patient_id"].unique())
ur = pd.DataFrame({"patient_id": patients,
                   "ur_number": [f"UR{n:07d}" for n in rng.choice(9_000_000, size=len(patients), replace=False) + 1_000_000]})
print(f"{len(src):,} resultados | {len(draws):,} muestras | {len(patients):,} pacientes")

### Paso 3 — Construir los resultados como los enviaría el laboratorio

**Explicación línea por línea**

- `for r in src.itertuples(index=False):` recorre cada resultado de Synthea; `itertuples` es mucho más rápido que `iterrows` y da acceso por atributo (`r.value_num`).
- `base = {...}`: campos comunes a cualquier resultado de esa muestra; `"status": "F"` es *final*, como el campo de estado del resultado en HL7 v2 (OBX-11).
- HbA1c (`4548-4`) genera **dos filas**: `A1C` en % con un decimal y `A1CI` en mmol/mol calculado con `10.93 * NGSP - 23.5`, la fórmula de la tabla de conversión de la Australian Diabetes Society. El armonizador usará después la **ecuación maestra del NGSP** (`NGSP = 0.09148 × IFCC + 2.152`); son dos formulaciones publicadas casi idénticas, y comprobarás que coinciden.
- `if r.old_analyser and local_code in OLD_ANALYSER:` el analizador antiguo informa el valor en mg/dL sin decimales.
- `text = f"{value:.{dec}f}"`: el resultado viaja como **texto** con los decimales que informa el laboratorio. **Por qué texto:** un resultado de laboratorio puede ser `5.2`, `<30` o `>500`; por eso en HL7 v2 y en los extractos reales el valor es una cadena.
- `if local_code == "CREAT" and value < 30: text = "<30"`: el analizador simulado no informa valores por debajo de 30 µmol/L (su límite inferior de medición); en su lugar envía un **resultado censurado**. El límite es parte de la simulación; el fenómeno (límites de cuantificación) es real.

In [ ]:
rows = []
for r in src.itertuples(index=False):
    base = {"ur_number": None, "patient_id": r.patient_id, "accession": r.accession, "collected_ts": r.obs_ts,
            "reported_ts": r.reported_ts, "status": "F"}
    if r.loinc_code == "4548-4":
        rows.append({**base, "local_code": "A1C", "local_name": "HbA1c (NGSP)", "specimen": "Blood",
                     "result": f"{r.value_num:.1f}", "units": "%"})
        rows.append({**base, "local_code": "A1CI", "local_name": "HbA1c (IFCC)", "specimen": "Blood",
                     "result": f"{10.93 * r.value_num - 23.5:.0f}", "units": "mmol/mol"})
        continue
    local_code, local_name, specimen, unit, factor, dec = SPEC[r.loinc_code]
    if r.old_analyser and local_code in OLD_ANALYSER:
        unit, value, dec = "mg/dL", r.value_num, 0
    else:
        value = r.value_num * factor
    text = f"{value:.{dec}f}"
    if local_code == "CREAT" and value < 30:
        text = "<30"
    rows.append({**base, "local_code": local_code, "local_name": local_name, "specimen": specimen,
                 "result": text, "units": unit})

feed = pd.DataFrame(rows).drop(columns="ur_number").merge(ur, on="patient_id", how="left", validate="many_to_one")
feed.groupby(["local_code", "units"]).size()

### Paso 4 — Inyectar los problemas típicos de un extracto real y guardarlo

**Explicación línea por línea**

- `lpa = ...sample(frac=0.05, random_state=7)`: en el 5 % de los perfiles lipídicos el laboratorio añade **lipoproteína(a)** con código `LPA`, una prueba que **no está en tu catálogo de mapeo**. `rng.lognormal(3.5, 1.0, ...)` genera valores positivos asimétricos (inventados). Servirá para la cola de códigos sin mapear.
- `corr = ...sample(frac=0.01, random_state=11)`: el 1 % de los resultados recibe una **corrección** (`status="C"`) un día después, con el valor ±5 %. `decimals(t)` cuenta los decimales del texto original para que la corrección se informe igual.
- `variant = rng.random(len(feed)) < 0.02`: en el 2 % de las filas el código llega en minúsculas o con espacios (`" chol"`, `"GLU  "`): un clásico de las interfaces.
- `def to_local_iso(ts):` convierte la hora UTC a la hora de Brisbane y la escribe en ISO 8601 **con desfase** (`+10:00`). Los laboratorios australianos envían hora local; si la tratas como UTC, desplazas todos los resultados 10 horas.
- `excluded = rng.choice(patients, size=10, replace=False)`: 10 pacientes del extracto **no** aparecen en la tabla de vinculación (`lab_ur_xref.csv`, que en un hospital vendría del índice maestro de pacientes, MPI). Sus resultados no podrán vincularse.
- `feed_out.to_csv(RAW_DIR / "lab_feed_extract.csv", index=False)`: el extracto queda en `data/raw`, como si hubiera llegado por SFTP. Recuerda: `data/` **nunca** se sube a git, aunque sea sintético.

In [ ]:
lpa = feed[feed["local_code"] == "CHOL"].sample(frac=0.05, random_state=7).copy()
lpa = lpa.assign(local_code="LPA", local_name="Lipoprotein (a)", units="nmol/L",
                 result=[f"{v:.0f}" for v in rng.lognormal(3.5, 1.0, len(lpa))])


def decimals(text):
    return len(text.split(".")[1]) if "." in text else 0


corr = feed[~feed["local_code"].isin(["A1C", "A1CI"]) & ~feed["result"].str.startswith("<")].sample(frac=0.01, random_state=11).copy()
corr["result"] = [f"{float(t) * (1 + d):.{decimals(t)}f}" for t, d in zip(corr["result"], rng.choice([-0.05, 0.05], len(corr)))]
corr["status"] = "C"
corr["reported_ts"] = corr["reported_ts"] + pd.Timedelta(days=1)

feed = pd.concat([feed, lpa, corr], ignore_index=True)
variant = rng.random(len(feed)) < 0.02
styles = rng.choice(["lower", "lpad", "rpad"], size=variant.sum())
feed.loc[variant, "local_code"] = [c.lower() if s == "lower" else (" " + c if s == "lpad" else c + "  ")
                                   for c, s in zip(feed.loc[variant, "local_code"], styles)]


def to_local_iso(ts):
    return ts.dt.tz_localize("UTC").dt.tz_convert("Australia/Brisbane").map(lambda t: t.isoformat())


feed_out = (feed.assign(lab_id="PATHLAB-SIM", collected_at=to_local_iso(feed["collected_ts"]),
                        reported_at=to_local_iso(feed["reported_ts"]))
                .sort_values(["reported_at", "accession", "local_code"])
                [["lab_id", "ur_number", "accession", "local_code", "local_name", "specimen", "result", "units",
                  "status", "collected_at", "reported_at"]])
excluded = rng.choice(patients, size=10, replace=False)
xref_out = ur[~ur["patient_id"].isin(excluded)]

feed_out.to_csv(RAW_DIR / "lab_feed_extract.csv", index=False)
xref_out.to_csv(RAW_DIR / "lab_ur_xref.csv", index=False)
print(f"Extracto: {len(feed_out):,} filas | vinculación: {len(xref_out):,} UR numbers")
feed_out.head(3)

## Parte A — Perfilar el extracto como lo recibes

### Paso 5 — Leer el extracto y priorizar por volumen

Desde aquí trabajas **solo** con los archivos de `data/raw`, como si no supieras cómo se generaron.

**Explicación línea por línea**

- `pd.read_csv(..., dtype=str, keep_default_na=False)`: todo se lee como texto y nada se convierte en `NaN` por sorpresa (por ejemplo, un código `"NA"`). Es la misma regla de la capa bronze de la semana 1.
- `raw["local_code"].unique()`: muestra los códigos tal como llegan; verás las variantes con espacios y minúsculas.
- `prof = raw.assign(code_norm=...)`: perfila por código normalizado y unidad.
- `prof["cum_share"] = prof["n"].cumsum() / prof["n"].sum()`: proporción acumulada del volumen. **Por qué importa:** Regenstrief recomienda empezar un mapeo grande por la lista *LOINC Top 2000+ Lab Observations*, que representa cerca del 98 % del volumen de pruebas de tres grandes organizaciones que mapearon todo su laboratorio (la lista está congelada desde LOINC 2.64). La lección se mantiene: en un proyecto real se mapea primero lo que más volumen tiene y se deja la cola larga para después.

In [ ]:
raw = pd.read_csv(RAW_DIR / "lab_feed_extract.csv", dtype=str, keep_default_na=False)
xref = pd.read_csv(RAW_DIR / "lab_ur_xref.csv", dtype=str, keep_default_na=False)
print("Códigos tal como llegan:", sorted(raw["local_code"].unique()))

prof = (raw.assign(code_norm=raw["local_code"].str.strip().str.upper())
           .groupby(["code_norm", "units"]).size().reset_index(name="n")
           .sort_values("n", ascending=False, ignore_index=True))
prof["cum_share"] = (prof["n"].cumsum() / prof["n"].sum()).round(3)
prof

## Parte B — Los seis ejes de LOINC

### Paso 6 — Tabla de referencia de los códigos de este proyecto

Cada código LOINC de laboratorio es la combinación de **seis ejes**: **Component** (qué se mide), **Property** (tipo de magnitud: `MCnc` concentración de masa, `SCnc` concentración de sustancia, `MFr` fracción de masa, `SFr` fracción de sustancia), **Time** (`Pt`, un punto en el tiempo), **System** (muestra: `Ser/Plas`, `Bld`), **Scale** (`Qn`, cuantitativa) y **Method** (opcional: `Direct assay`, `IFCC`). Por eso el colesterol en mg/dL (`2093-3`, MCnc) y en mmol/L (`14647-2`, SCnc) son **códigos distintos**: cambia la propiedad. En FHIR, el sistema de códigos LOINC expone esos ejes como las propiedades `COMPONENT`, `PROPERTY`, `TIME_ASPCT`, `SYSTEM`, `SCALE_TYP` y `METHOD_TYP`.

**Explicación línea por línea**

- `LOINC_REF = pd.DataFrame([...], columns=[...])`: una fila por código con su nombre largo común (*Long Common Name*) y sus seis ejes. Son los siete códigos que usa Synthea más sus equivalentes SI. **En la práctica real** esta tabla sale de la distribución oficial de LOINC (`Loinc.csv`) o del buscador de LOINC/RELMA; aquí la escribes a mano para un alcance pequeño y la verificas contra el servidor en el paso siguiente.
- Observa `18262-6`: LDL **medido directamente** (`Direct assay`) y en masa; `22748-8`: LDL en sustancia, sin método. Y `4548-4` (NGSP, %, MFr) frente a `59261-8` (IFCC, mmol/mol, SFr). Según LOINC, `4548-4` es el código para informar HbA1c en EE. UU. y otros países pueden informar por protocolo IFCC con `59261-8`.

In [ ]:
LOINC_REF = pd.DataFrame(
    [
        ("2093-3", "Cholesterol [Mass/volume] in Serum or Plasma", "Cholesterol", "MCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("14647-2", "Cholesterol [Moles/volume] in Serum or Plasma", "Cholesterol", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("2085-9", "Cholesterol in HDL [Mass/volume] in Serum or Plasma", "Cholesterol.in HDL", "MCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("14646-4", "Cholesterol in HDL [Moles/volume] in Serum or Plasma", "Cholesterol.in HDL", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("18262-6", "Cholesterol in LDL [Mass/volume] in Serum or Plasma by Direct assay", "Cholesterol.in LDL", "MCnc", "Pt", "Ser/Plas", "Qn", "Direct assay"),
        ("22748-8", "Cholesterol in LDL [Moles/volume] in Serum or Plasma", "Cholesterol.in LDL", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("2571-8", "Triglyceride [Mass/volume] in Serum or Plasma", "Triglyceride", "MCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("14927-8", "Triglyceride [Moles/volume] in Serum or Plasma", "Triglyceride", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("2339-0", "Glucose [Mass/volume] in Blood", "Glucose", "MCnc", "Pt", "Bld", "Qn", ""),
        ("15074-8", "Glucose [Moles/volume] in Blood", "Glucose", "SCnc", "Pt", "Bld", "Qn", ""),
        ("14749-6", "Glucose [Moles/volume] in Serum or Plasma", "Glucose", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("38483-4", "Creatinine [Mass/volume] in Blood", "Creatinine", "MCnc", "Pt", "Bld", "Qn", ""),
        ("59826-8", "Creatinine [Moles/volume] in Blood", "Creatinine", "SCnc", "Pt", "Bld", "Qn", ""),
        ("14682-9", "Creatinine [Moles/volume] in Serum or Plasma", "Creatinine", "SCnc", "Pt", "Ser/Plas", "Qn", ""),
        ("4548-4", "Hemoglobin A1c/Hemoglobin.total in Blood", "Hemoglobin A1c/Hemoglobin.total", "MFr", "Pt", "Bld", "Qn", ""),
        ("59261-8", "Hemoglobin A1c/Hemoglobin.total in Blood by IFCC protocol", "Hemoglobin A1c/Hemoglobin.total", "SFr", "Pt", "Bld", "Qn", "IFCC"),
    ],
    columns=["loinc_code", "long_common_name", "component", "property", "time", "system", "scale", "method"],
)

LOINC_REF

### Paso 7 — (En línea) Verificar los ejes contra el NCTS con `$lookup`

El NCTS publica LOINC como sistema de códigos FHIR. Este paso reutiliza el patrón de credenciales y token de la semana 6.

**Explicación línea por línea**

- `get_secret`, `get_token`, `tx_get`, `params_to_dict`: las mismas funciones de la semana 6 (credenciales en Colab Secrets, token OAuth2 *client credentials* con caché, `GET` autenticado y lectura del recurso `Parameters`). Se repiten para que este notebook sea autosuficiente.
- `LOINC_PROPS = ["COMPONENT", "PROPERTY", "TIME_ASPCT", "SYSTEM", "SCALE_TYP", "METHOD_TYP"]`: los ejes como propiedades FHIR de LOINC.
- `tx_get("CodeSystem/$lookup", [("system", "http://loinc.org"), ("code", c)] + [("property", p) for p in LOINC_PROPS])`: una lista de tuplas permite repetir el parámetro `property` (no se puede con un diccionario).
- `def prop_text(v):` el valor de una propiedad puede llegar como `Coding` (una "parte" LOINC con código `LP...` y su `display`) o como texto; se toma el texto legible.
- `LOINC_VERSION = res.get("version") or LOINC_VERSION`: guarda la versión de LOINC que tiene el servidor; quedará registrada en el *crosswalk*. Sin conexión se usa `"2.82"`, la versión que el NCTS anunció como disponible el 20 de marzo de 2026; con conexión se registra la que informe el servidor en ese momento.
- `time.sleep(0.2)`: ritmo controlado, igual que en la semana 6.
- La comparación es **informativa**: el servidor puede devolver los nombres de las partes con otra ortografía; lo importante es que revises cualquier diferencia de propiedad o sistema.

In [ ]:
import requests

NCTS_TOKEN_URL = os.environ.get("NCTS_TOKEN_URL", "https://api.healthterminologies.gov.au/oauth2/token")
NCTS_FHIR_BASE = os.environ.get("NCTS_FHIR_BASE", "https://api.healthterminologies.gov.au/integration/R4/fhir")


def get_secret(name):
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get(name)
        except Exception as exc:
            print(f"Secreto {name} no disponible ({type(exc).__name__})")
            return None
    return os.environ.get(name)


CLIENT_ID = get_secret("NCTS_CLIENT_ID")
CLIENT_SECRET = get_secret("NCTS_CLIENT_SECRET")
ONLINE = bool(CLIENT_ID and CLIENT_SECRET)
_token = {"value": None, "expires_at": 0.0}
session = requests.Session()
session.headers["Accept"] = "application/fhir+json"


def get_token():
    if _token["value"] and time.time() < _token["expires_at"] - 60:
        return _token["value"]
    r = requests.post(NCTS_TOKEN_URL, data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
                                            "client_secret": CLIENT_SECRET}, timeout=30)
    r.raise_for_status()
    payload = r.json()
    _token["value"] = payload["access_token"]
    _token["expires_at"] = time.time() + float(payload.get("expires_in", 300))
    return _token["value"]


def tx_get(path, params):
    r = session.get(f"{NCTS_FHIR_BASE}/{path}", params=params,
                    headers={"Authorization": f"Bearer {get_token()}"}, timeout=60)
    r.raise_for_status()
    return r.json()


def _value(p):
    return next((v for k, v in p.items() if k.startswith("value")), None)


def params_to_dict(parameters):
    out = {"designation": [], "property": {}}
    for p in parameters.get("parameter", []):
        name = p.get("name")
        if name == "designation":
            out["designation"].append({q["name"]: _value(q) for q in p.get("part", [])})
        elif name == "property":
            parts = {q["name"]: _value(q) for q in p.get("part", [])}
            out["property"].setdefault(parts.get("code"), []).append(parts.get("value"))
        else:
            out[name] = _value(p)
    return out


print("Modo en línea con NCTS:", ONLINE)

In [ ]:
LOINC_PROPS = ["COMPONENT", "PROPERTY", "TIME_ASPCT", "SYSTEM", "SCALE_TYP", "METHOD_TYP"]
LOINC_VERSION = "2.82"


def prop_text(v):
    if isinstance(v, dict):
        return v.get("display") or v.get("code")
    return v


if ONLINE:
    found = []
    for c in LOINC_REF["loinc_code"]:
        res = params_to_dict(tx_get("CodeSystem/$lookup", [("system", "http://loinc.org"), ("code", c)]
                                    + [("property", p) for p in LOINC_PROPS]))
        LOINC_VERSION = res.get("version") or LOINC_VERSION
        found.append({"loinc_code": c, "display_ncts": res.get("display"),
                      **{p: prop_text((res["property"].get(p) or [None])[0]) for p in LOINC_PROPS}})
        time.sleep(0.2)
    axes_check = LOINC_REF.merge(pd.DataFrame(found), on="loinc_code")
    print("Versión de LOINC en el servidor:", LOINC_VERSION)
    display(axes_check[["loinc_code", "property", "PROPERTY", "system", "SYSTEM", "method", "METHOD_TYP"]])
else:
    print("Sin credenciales: se omite $lookup. Versión de LOINC registrada:", LOINC_VERSION)

## Parte C — Mapeo semiautomático de códigos locales a LOINC

### Paso 8 — Inferir los ejes desde lo que envía el laboratorio

Herramientas como RELMA (de Regenstrief) ayudan a mapear proponiendo candidatos a partir del nombre local, la muestra y la unidad; **la decisión final siempre es humana**. Aquí implementas la misma lógica, de forma transparente.

**Explicación línea por línea**

- `COMPONENT_RULES`: palabras clave del nombre local → componente LOINC. **El orden importa:** `"hdl"` y `"ldl"` van antes que `"cholesterol"`, porque "HDL Cholesterol" contiene la palabra "cholesterol".
- `PROPERTY_BY_UNIT`: la unidad delata la propiedad: mmol/L y µmol/L → `SCnc`, mg/dL → `MCnc`, % → `MFr`, mmol/mol → `SFr`.
- `AMBIGUOUS_UNITS = {"%"}`: "%" puede ser fracción de masa, de volumen o de número según la prueba (el hematocrito, por ejemplo, también es %). Un mapeo inferido desde "%" **nunca** se acepta automáticamente.
- `SYSTEM_BY_SPECIMEN`: muestra local → eje System (`Serum` y `Plasma` → `Ser/Plas`; `Blood` → `Bld`).
- `METHOD_RULES`: el método solo se asigna si el nombre local lo dice ("direct", "ifcc").
- `def infer_axes(local_name, specimen, unit):` devuelve un diccionario con los seis ejes; `next((... for ...), None)` toma la primera regla que coincide o `None` si ninguna.
- `def match_loinc(axes, unit, ref):`
  - si falta cualquier eje obligatorio → `"unmapped"`;
  - filtra la referencia eje por eje (`cand = cand[cand[k] == axes[k]]`);
  - si un único candidato coincide también en método → `"auto"` (o `"needs_review"` si la unidad es ambigua);
  - si coinciden los cinco ejes pero no el método → `"needs_review"`: el código existe, pero alguien debe confirmar el método;
  - en otro caso → `"unmapped"`.

In [ ]:
COMPONENT_RULES = [
    ("hdl", "Cholesterol.in HDL"),
    ("ldl", "Cholesterol.in LDL"),
    ("cholesterol", "Cholesterol"),
    ("triglycer", "Triglyceride"),
    ("glucose", "Glucose"),
    ("creatinine", "Creatinine"),
    ("hba1c", "Hemoglobin A1c/Hemoglobin.total"),
]
PROPERTY_BY_UNIT = {"mmol/L": "SCnc", "umol/L": "SCnc", "mg/dL": "MCnc", "%": "MFr", "mmol/mol": "SFr"}
AMBIGUOUS_UNITS = {"%"}
SYSTEM_BY_SPECIMEN = {"serum": "Ser/Plas", "plasma": "Ser/Plas", "blood": "Bld"}
METHOD_RULES = [("direct", "Direct assay"), ("ifcc", "IFCC")]
AXES = ["component", "property", "time", "system", "scale"]


def infer_axes(local_name, specimen, unit):
    name = local_name.lower()
    return {
        "component": next((c for key, c in COMPONENT_RULES if key in name), None),
        "property": PROPERTY_BY_UNIT.get(unit),
        "time": "Pt",
        "system": SYSTEM_BY_SPECIMEN.get(specimen.strip().lower()),
        "scale": "Qn",
        "method": next((m for key, m in METHOD_RULES if key in name), ""),
    }


def match_loinc(axes, unit, ref):
    if any(axes[k] is None for k in AXES):
        return None, "unmapped"
    cand = ref
    for k in AXES:
        cand = cand[cand[k] == axes[k]]
    exact = cand[cand["method"] == axes["method"]]
    if len(exact) == 1:
        return exact.iloc[0]["loinc_code"], ("needs_review" if unit in AMBIGUOUS_UNITS else "auto")
    if len(cand) == 1:
        return cand.iloc[0]["loinc_code"], "needs_review"
    return None, "unmapped"

### Paso 9 — Construir el *crosswalk* (tabla de equivalencias)

**Explicación línea por línea**

- `catalog = raw.assign(local_code=...).groupby([...]).size()`: una fila por combinación **código local + nombre + muestra + unidad**. **Por qué la unidad forma parte de la clave:** `CHOL` en mmol/L es `14647-2`, pero `CHOL` en mg/dL (el analizador antiguo) es `2093-3`. El mismo código local puede corresponder a dos LOINC distintos.
- `for r in catalog.itertuples(index=False):` aplica `infer_axes` y `match_loinc` a cada fila; `r._asdict()` convierte la tupla en diccionario para combinarla con los ejes.
- `.merge(LOINC_REF[["loinc_code", "long_common_name"]], ...)`: añade el nombre LOINC para que el revisor lea algo comprensible.
- `crosswalk["status"].value_counts()`: cuántos quedaron `auto`, `needs_review` y `unmapped`.

In [ ]:
catalog = (raw.assign(local_code=raw["local_code"].str.strip().str.upper())
              .groupby(["local_code", "local_name", "specimen", "units"]).size()
              .reset_index(name="n_results"))
records = []
for r in catalog.itertuples(index=False):
    axes = infer_axes(r.local_name, r.specimen, r.units)
    loinc_code, status = match_loinc(axes, r.units, LOINC_REF)
    records.append({**r._asdict(), **axes, "loinc_code": loinc_code, "status": status})
crosswalk = pd.DataFrame(records).merge(LOINC_REF[["loinc_code", "long_common_name"]], on="loinc_code", how="left")
print(crosswalk["status"].value_counts().to_dict())
crosswalk[["local_code", "units", "n_results", "property", "system", "method", "loinc_code", "status"]]

### Paso 10 — Revisión humana, versión y cola de pendientes

**Explicación línea por línea**

- `REVIEWER = "revisor_demo"`: en un proyecto real son las iniciales del patólogo químico o del terminólogo que firma el mapeo. **Nunca** se carga a producción un mapeo sin responsable.
- `REVIEW_DATE = "2026-11-22"`: fecha de la **decisión**, fija. Si re-ejecutas el notebook, la decisión no cambió y la tabla tampoco debe cambiar (su huella SHA-256 sirve como versión).
- `DECISIONS = {...}`: las decisiones de los casos que la regla no resolvió sola, con su justificación:
  - `("A1C", "%")`: el laboratorio confirma que el % es HbA1c estandarizada NGSP (fracción de masa) → `4548-4`;
  - `("LDL", "mg/dL")`: el laboratorio confirma que el analizador antiguo medía LDL **directo** → `18262-6`.
  En la simulación, "el laboratorio confirma" es una suposición documentada; en la vida real es un correo o una reunión que se archiva.
- `auto = crosswalk["status"] == "auto"`: las coincidencias exactas también se aprueban, pero quedan registradas como tales en `review_note`.
- `crosswalk["loinc_version"] = LOINC_VERSION`: la versión de LOINC contra la que se mapeó. **Por qué:** LOINC publica versiones dos veces al año y puede desaprobar códigos; sin versión no puedes auditar el mapeo.
- `crosswalk["equivalence"] = ...`: el grado de equivalencia con el vocabulario de `ConceptMap` de FHIR R4: `equivalent` para lo aprobado (mismo significado) y `unmatched` para lo que no tiene destino. Lo usarás en la Parte I.
- `queue = crosswalk[crosswalk["status"] != "approved"]`: lo que queda (aquí, `LPA`) va a una **cola de revisión** con su volumen, para que el terminólogo lo busque en el buscador de LOINC o en RELMA.
- `crosswalk.to_csv(REF_DIR / "loinc_crosswalk.csv", index=False)`: el *crosswalk* es **metadato, no dato de pacientes**: en un proyecto real se versiona en git (en la semana 10 lo empaquetarás dentro del pipeline).

In [ ]:
REVIEWER = "revisor_demo"
REVIEW_DATE = "2026-11-22"
DECISIONS = {
    ("A1C", "%"): ("4548-4", "Laboratorio confirma HbA1c NGSP; % = fracción de masa"),
    ("LDL", "mg/dL"): ("18262-6", "Laboratorio confirma LDL por medición directa en el analizador antiguo"),
}

crosswalk["reviewer"] = None
crosswalk["review_date"] = None
crosswalk["review_note"] = None
auto = crosswalk["status"] == "auto"
crosswalk.loc[auto, ["status", "reviewer", "review_date", "review_note"]] = ["approved", REVIEWER, REVIEW_DATE,
                                                                             "Coincidencia exacta de ejes y método"]
for (lc, unit), (loinc_code, note) in DECISIONS.items():
    sel = (crosswalk["local_code"] == lc) & (crosswalk["units"] == unit) & (crosswalk["status"] == "needs_review")
    assert sel.sum() == 1 and (crosswalk.loc[sel, "loinc_code"] == loinc_code).all(), (lc, unit)
    crosswalk.loc[sel, ["status", "reviewer", "review_date", "review_note"]] = ["approved", REVIEWER, REVIEW_DATE, note]
crosswalk["loinc_version"] = LOINC_VERSION
crosswalk["equivalence"] = crosswalk["status"].map({"approved": "equivalent"}).fillna("unmatched")

queue = crosswalk.loc[crosswalk["status"] != "approved", ["local_code", "local_name", "specimen", "units", "n_results", "status"]]
crosswalk.to_csv(REF_DIR / "loinc_crosswalk.csv", index=False)
queue.to_csv(REF_DIR / "loinc_unmapped_queue.csv", index=False)
print(crosswalk["status"].value_counts().to_dict())
queue

## Parte D — Validar los códigos

### Paso 11 — Dígito verificador y tipos de código LOINC

**Explicación línea por línea**

- `def loinc_check_digit(body):` y `def loinc_ok(code):` son el algoritmo Mod 10 de la semana 6 para códigos de **término** (`14647-2`).
- `def loinc_kind(code):` LOINC usa también códigos de **respuesta** (`LA` + dígitos + guion + dígito, para resultados codificados como "Positivo") y de **parte** (`LP...`, los componentes de los ejes que viste en `$lookup`). En nuestras pruebas, la fórmula Mod 10 de los términos **no** se cumplió para códigos `LA`; por eso, para `LA` y `LP` solo se valida el **formato** con `re.fullmatch` y la existencia se confirma en el servidor. **Lección práctica:** el validador de la semana 6 era correcto para términos, pero aplicarlo a cualquier cadena que "parezca LOINC" produciría falsos errores.
- `assert all(loinc_ok(c) for c in LOINC_REF["loinc_code"])`: todos los términos de tu referencia pasan el dígito verificador; si copiaras mal un código, el notebook se detendría aquí.

In [ ]:
def loinc_check_digit(body):
    digits = [int(ch) for ch in body][::-1]
    total = 0
    for i, d in enumerate(digits):
        if i % 2 == 0:
            d *= 2
            d = d - 9 if d > 9 else d
        total += d
    return (10 - total % 10) % 10


def loinc_ok(code):
    if not isinstance(code, str) or not re.fullmatch(r"\d+-\d", code):
        return False
    body, check = code.split("-")
    return loinc_check_digit(body) == int(check)


def loinc_kind(code):
    if re.fullmatch(r"\d+-\d", code):
        return "term" if loinc_ok(code) else "term_bad_check_digit"
    if re.fullmatch(r"LA\d+-\d", code):
        return "answer_format_ok"
    if re.fullmatch(r"LP\d+-\d", code):
        return "part_format_ok"
    return "not_loinc"


assert all(loinc_ok(c) for c in LOINC_REF["loinc_code"])
assert loinc_kind("14647-2") == "term" and loinc_kind("14647-3") == "term_bad_check_digit"
assert loinc_kind("QALY") == "not_loinc"
print("Dígitos verificadores OK para", len(LOINC_REF), "códigos de referencia")

### Paso 12 — (En línea) `$validate-code` y conjuntos de referencia SPIA del RCPA

En Australia, el RCPA publica los conjuntos de referencia SPIA, que definen qué códigos LOINC y qué unidades se esperan en los informes de patología; el NCTS los distribuye como `ValueSet` FHIR.

**Explicación línea por línea**

- `SPIA_CHEM = "https://www.rcpa.edu.au/fhir/ValueSet/spia-chemical-pathology-refset-3"` y `SPIA_UNITS = ".../spia-preferred-units-refset-3"`: URL canónicas de los conjuntos de química clínica y de unidades preferidas.
- `tx_get("CodeSystem/$validate-code", {"url": "http://loinc.org", "code": c})`: ¿existe el código en LOINC?
- `tx_get("ValueSet/$validate-code", {"url": SPIA_CHEM, "system": "http://loinc.org", "code": c})`: ¿está en el conjunto SPIA? Un código válido en LOINC pero fuera de SPIA no es un error de LOINC: es una **conversación con el laboratorio** (quizá deba informar otro código).
- Para unidades: `system=http://unitsofmeasure.org` y `code=<unidad UCUM>` contra `SPIA_UNITS`.
- `.get("result")` y `.get("message")`: la respuesta es un `Parameters` con `result` (verdadero/falso) y, si falla, un `message`.
- Se valida lo **aprobado** en el crosswalk: lo que de verdad entrará a tus tablas.

In [ ]:
SPIA_CHEM = "https://www.rcpa.edu.au/fhir/ValueSet/spia-chemical-pathology-refset-3"
SPIA_UNITS = "https://www.rcpa.edu.au/fhir/ValueSet/spia-preferred-units-refset-3"
approved = crosswalk[crosswalk["status"] == "approved"]

if ONLINE:
    checks = []
    for c in sorted(approved["loinc_code"].unique()):
        in_loinc = params_to_dict(tx_get("CodeSystem/$validate-code", {"url": "http://loinc.org", "code": c}))
        in_spia = params_to_dict(tx_get("ValueSet/$validate-code", {"url": SPIA_CHEM, "system": "http://loinc.org", "code": c}))
        checks.append({"code": c, "kind": "loinc", "in_loinc": in_loinc.get("result"), "in_spia": in_spia.get("result"),
                       "message": in_spia.get("message")})
        time.sleep(0.2)
    for u in sorted(approved["units"].unique()):
        in_units = params_to_dict(tx_get("ValueSet/$validate-code", {"url": SPIA_UNITS, "system": "http://unitsofmeasure.org", "code": u}))
        checks.append({"code": u, "kind": "unit", "in_loinc": None, "in_spia": in_units.get("result"), "message": in_units.get("message")})
        time.sleep(0.2)
    spia_report = pd.DataFrame(checks)
    spia_report.to_csv(REPORTS_DIR / "loinc_spia_validation.csv", index=False)
    display(spia_report)
else:
    print("Sin credenciales: se omite la validación en línea (LOINC y SPIA).")

## Parte E — Armonizar unidades a SI con UCUM

### Paso 13 — Conversiones documentadas y probadas

**Explicación línea por línea**

- `ANALYTE_BY_COMPONENT`: nombre analítico corto por componente LOINC; servirá para unir fuentes que usan **códigos distintos para el mismo analito** (`2093-3` y `14647-2` son ambos `total_cholesterol`).
- `SI_UNIT`: la unidad SI de destino de cada analito, escrita en **UCUM** (`umol/L` es la forma UCUM de µmol/L). UCUM es el estándar que exige FHIR para cantidades (`valueQuantity.system = http://unitsofmeasure.org`).
- `MOLAR_MASS_G_PER_MOL`: masas molares que LOINC Groups publica para colesterol (386,664 g/mol), glucosa (180,156 g/mol) y creatinina (113,12 g/mol).
- `TRIGLYCERIDE_MGDL_TO_MMOLL = 0.01129`: los triglicéridos son una mezcla de moléculas sin masa molar única; se usa el factor convencional (NHANES).
- `def to_si(component, value, unit):`
  - si ya está en la unidad destino, devuelve el valor;
  - mg/dL → mmol/L: `value * 10 / masa_molar` (el `* 10` pasa de dL a L; dividir por g/mol pasa de mg a mmol); para creatinina se multiplica por 1000 para obtener µmol/L (1 mg/dL ≈ 88,4 µmol/L);
  - HbA1c % → mmol/mol: `(NGSP − 2.152) / 0.09148`, despejando la ecuación maestra del NGSP;
  - cualquier otra combinación → `NaN` y unidad `None`: **una conversión desconocida nunca se inventa**.
- Los `assert` comprueban anclas conocidas: 6,5 % de HbA1c ≈ 48 mmol/mol; 1 mg/dL de creatinina ≈ 88,4 µmol/L; 90 mg/dL de glucosa ≈ 5,0 mmol/L.

In [ ]:
ANALYTE_BY_COMPONENT = {
    "Cholesterol": "total_cholesterol",
    "Cholesterol.in HDL": "hdl_cholesterol",
    "Cholesterol.in LDL": "ldl_cholesterol",
    "Triglyceride": "triglycerides",
    "Glucose": "glucose",
    "Creatinine": "creatinine",
    "Hemoglobin A1c/Hemoglobin.total": "hba1c",
}
SI_UNIT = {"total_cholesterol": "mmol/L", "hdl_cholesterol": "mmol/L", "ldl_cholesterol": "mmol/L",
           "triglycerides": "mmol/L", "glucose": "mmol/L", "creatinine": "umol/L", "hba1c": "mmol/mol"}
MOLAR_MASS_G_PER_MOL = {"Cholesterol": 386.664, "Cholesterol.in HDL": 386.664, "Cholesterol.in LDL": 386.664,
                        "Glucose": 180.156, "Creatinine": 113.12}
TRIGLYCERIDE_MGDL_TO_MMOLL = 0.01129


def to_si(component, value, unit):
    analyte = ANALYTE_BY_COMPONENT.get(component)
    if analyte is None or pd.isna(value):
        return float("nan"), None
    target = SI_UNIT[analyte]
    if unit == target:
        return float(value), target
    if unit == "mg/dL" and component in MOLAR_MASS_G_PER_MOL:
        mmol_per_l = value * 10 / MOLAR_MASS_G_PER_MOL[component]
        return (mmol_per_l * 1000 if target == "umol/L" else mmol_per_l), target
    if unit == "mg/dL" and component == "Triglyceride":
        return value * TRIGLYCERIDE_MGDL_TO_MMOLL, target
    if unit == "%" and analyte == "hba1c":
        return (value - 2.152) / 0.09148, target
    return float("nan"), None


assert round(to_si("Hemoglobin A1c/Hemoglobin.total", 6.5, "%")[0]) == 48
assert abs(to_si("Creatinine", 1.0, "mg/dL")[0] - 88.4) < 0.01
assert abs(to_si("Glucose", 90, "mg/dL")[0] - 5.0) < 0.01
assert np.isnan(to_si("Glucose", 90, "g/L")[0])
print("Conversiones OK")

### Paso 14 — Leer resultados como texto: comparadores y censura

**Explicación línea por línea**

- `RESULT_RE = re.compile(r"(<=|>=|<|>)?\s*(-?\d+(?:\.\d+)?)")`: un comparador opcional (los de dos caracteres primero, para que `<=` no se lea como `<`), espacios opcionales y un número con signo y decimales opcionales.
- `RESULT_RE.fullmatch(str(text).strip())`: toda la cadena debe ajustarse al patrón; `"5.2 mmol"` o `"hemolizada"` no se aceptan como número.
- `return (m.group(1) or "="), float(m.group(2))`: devuelve el comparador (`"="` si no hay) y el número. **Por qué separar comparador y valor:** `<30` no es 30. Si lo guardas como 30 sin más, sesgas los promedios; si lo descartas, pierdes la información de que el valor era bajo. FHIR lo representa con `valueQuantity.comparator` y OMOP con `operator_concept_id`.

In [ ]:
RESULT_RE = re.compile(r"(<=|>=|<|>)?\s*(-?\d+(?:\.\d+)?)")


def parse_result(text):
    m = RESULT_RE.fullmatch(str(text).strip())
    if not m:
        return None, float("nan")
    return (m.group(1) or "="), float(m.group(2))


assert parse_result("<30") == ("<", 30.0)
assert parse_result(" 5.2 ") == ("=", 5.2)
assert parse_result(">=7") == (">=", 7.0)
assert parse_result("hemolizada")[0] is None
print("Lectura de resultados OK")

### Paso 15 — Limpiar el extracto: códigos, hora local y resultados corregidos

**Explicación línea por línea**

- `f["local_code"].str.strip().str.upper()`: normaliza `" chol"` → `CHOL`.
- `parsed = [parse_result(t) for t in f["result"]]`: comparador y valor de cada resultado.
- `pd.to_datetime(..., utc=True, format="ISO8601")`: interpreta el desfase `+10:00` y convierte a UTC; `.dt.tz_localize(None)` deja la hora UTC sin zona (la convención de silver desde la semana 3). `.astype("datetime64[us]")` iguala la resolución a la de silver para poder comparar horas exactamente.
- `sort_values([... "reported_ts"], kind="stable")` y `drop_duplicates(["accession", "local_code"], keep="last")`: para una misma muestra y prueba, gana el **último informe**. Así una corrección (`C`) reemplaza al final original (`F`), que es lo que establece el estado C en HL7 v2.
- Devuelve el extracto limpio y cuántas filas fueron reemplazadas.

In [ ]:
def clean_feed(feed):
    f = feed.copy()
    f["local_code"] = f["local_code"].str.strip().str.upper()
    parsed = [parse_result(t) for t in f["result"]]
    f["comparator"] = [p[0] for p in parsed]
    f["value_num"] = [p[1] for p in parsed]
    for col in ["collected", "reported"]:
        ts = pd.to_datetime(f[f"{col}_at"], utc=True, format="ISO8601")
        f[f"{col}_ts"] = ts.dt.tz_localize(None).astype("datetime64[us]")
    f = f.sort_values(["accession", "local_code", "reported_ts"], kind="stable")
    n_before = len(f)
    f = f.drop_duplicates(["accession", "local_code"], keep="last").reset_index(drop=True)
    return f, n_before - len(f)


clean, n_replaced = clean_feed(raw)
print(f"Filas: {len(raw):,} → {len(clean):,} ({n_replaced} reemplazadas por correcciones)")
print("Códigos tras limpiar:", sorted(clean["local_code"].unique()))

### Paso 16 — Vincular pacientes, aplicar el crosswalk y poner en cuarentena

**Explicación línea por línea**

- `feed.merge(xref, on="ur_number", how="left", validate="many_to_one")`: añade `patient_id`. `validate="many_to_one"` hace que pandas lance un error si un UR number apareciera dos veces en la tabla de vinculación (eso duplicaría resultados sin que lo notes).
- `approved = crosswalk.loc[crosswalk["status"] == "approved", ...]`: **solo** lo aprobado se aplica.
- `merge(approved, on=["local_code", "units"], ...)`: el LOINC depende del código **y** de la unidad.
- `quarantine_reason`: `unmapped_code` si no hay LOINC aprobado y `unlinked_ur_number` si no hay paciente. Las filas en cuarentena **no se borran**: se guardan en un reporte para que alguien las resuelva. **En la práctica real**, perder resultados de laboratorio en silencio es uno de los errores más graves de un ETL clínico.

In [ ]:
def link_and_map(feed, xref, crosswalk):
    linked = feed.merge(xref, on="ur_number", how="left", validate="many_to_one")
    approved = crosswalk.loc[crosswalk["status"] == "approved", ["local_code", "units", "loinc_code"]]
    mapped = linked.merge(approved, on=["local_code", "units"], how="left", validate="many_to_one")
    reason = pd.Series(pd.NA, index=mapped.index, dtype="object")
    reason[mapped["loinc_code"].isna()] = "unmapped_code"
    reason[mapped["patient_id"].isna()] = "unlinked_ur_number"
    mapped["quarantine_reason"] = reason
    return mapped


mapped = link_and_map(clean, xref, crosswalk)
quarantine = mapped[mapped["quarantine_reason"].notna()]
quarantine.to_csv(REPORTS_DIR / "lab_quarantine.csv", index=False)
print(quarantine["quarantine_reason"].value_counts().to_dict())

### Paso 17 — Dos fuentes, un mismo formato, y conversión a SI

**Explicación línea por línea**

- `LAB_COLUMNS`: el esquema común de resultados de laboratorio. Conserva **lo que dijo la fuente** (`source_code`, `source_unit`, `source_value_text`) junto a lo estandarizado (`loinc_code`, `value_si`, `unit_si`). Es el mismo principio de OMOP: valor de origen + concepto estándar.
- `def ehr_lab_rows(obs, ref):` toma de silver los resultados con LOINC de la referencia y valor numérico; `source="synthea_ehr"`; el `encounter_id` hace de identificador de origen.
- `def feed_lab_rows(mapped):` toma solo lo que no está en cuarentena; `source="pathlab_feed"`, `source_code_system="PATHLAB_SIM"` (el "vocabulario" local del laboratorio); `status` se traduce de HL7 v2 (`F`, `C`) a los códigos de FHIR (`final`, `corrected`).
- `def harmonize(rows, ref):` busca el componente de cada LOINC, aplica `to_si` y añade `analyte`, `value_si` y `unit_si`.
- `assert lab["unit_si"].notna().all()`: toda fila numérica debe tener conversión conocida; si apareciera una unidad nueva, el notebook se detiene en vez de guardar basura.

In [ ]:
LAB_COLUMNS = ["patient_id", "source", "source_record_id", "encounter_id", "collected_ts", "source_code",
               "source_code_system", "source_unit", "source_value_text", "comparator", "value_num_source",
               "loinc_code", "status"]


def ehr_lab_rows(obs, ref):
    e = obs[obs["loinc_code"].isin(ref["loinc_code"]) & obs["value_num"].notna()]
    return pd.DataFrame({
        "patient_id": e["patient_id"], "source": "synthea_ehr", "source_record_id": e["encounter_id"],
        "encounter_id": e["encounter_id"], "collected_ts": e["obs_ts"].astype("datetime64[us]"),
        "source_code": e["loinc_code"], "source_code_system": "LOINC", "source_unit": e["unit"],
        "source_value_text": e["value_num"].astype(str),"comparator": "=", "value_num_source": e["value_num"],
        "loinc_code": e["loinc_code"], "status": "final",
    })[LAB_COLUMNS]


def feed_lab_rows(mapped):
    m = mapped[mapped["quarantine_reason"].isna()]
    return pd.DataFrame({
        "patient_id": m["patient_id"], "source": "pathlab_feed", "source_record_id": m["accession"],
        "encounter_id": None, "collected_ts": m["collected_ts"], "source_code": m["local_code"],
        "source_code_system": "PATHLAB_SIM", "source_unit": m["units"], "source_value_text": m["result"],
        "comparator": m["comparator"], "value_num_source": m["value_num"], "loinc_code": m["loinc_code"],
        "status": m["status"].map({"F": "final", "C": "corrected"}),
    })[LAB_COLUMNS]


def harmonize(rows, ref):
    component = rows["loinc_code"].map(ref.set_index("loinc_code")["component"])
    conv = [to_si(c, v, u) for c, v, u in zip(component, rows["value_num_source"], rows["source_unit"])]
    out = rows.assign(analyte=component.map(ANALYTE_BY_COMPONENT),
                      value_si=[c[0] for c in conv], unit_si=[c[1] for c in conv])
    return out.reset_index(drop=True)


ehr = harmonize(ehr_lab_rows(obs, LOINC_REF), LOINC_REF)
lab_feed = harmonize(feed_lab_rows(mapped), LOINC_REF)
lab = pd.concat([ehr, lab_feed], ignore_index=True)
assert lab["unit_si"].notna().all(), lab[lab["unit_si"].isna()].head()
lab.groupby(["source", "loinc_code", "source_unit", "unit_si"]).size()

### Paso 18 — Consistencia de la HbA1c informada en dos unidades

**Explicación línea por línea**

- `a1c = lab_feed[lab_feed["analyte"] == "hba1c"].pivot_table(index="source_record_id", columns="loinc_code", values="value_si")`: una fila por muestra, con el valor convertido desde % (`4548-4`) y el informado en mmol/mol (`59261-8`).
- `diff = (a1c["4548-4"] - a1c["59261-8"]).abs()`: si ambas cifras describen la misma medición, deben coincidir salvo redondeo (el laboratorio informa mmol/mol sin decimales).
- `assert diff.max() <= 1`: tolerancia de 1 mmol/mol. **En la práctica real** esta prueba detecta interfaces que intercambian unidades o fórmulas mal configuradas en el LIS.

In [ ]:
a1c = lab_feed[lab_feed["analyte"] == "hba1c"].pivot_table(index="source_record_id", columns="loinc_code", values="value_si")
diff = (a1c["4548-4"] - a1c["59261-8"]).abs()
print(f"Muestras con HbA1c doble: {len(a1c):,} | diferencia máxima: {diff.max():.2f} mmol/mol")
assert diff.max() <= 1

## Parte F — Prueba de conformidad con los datos oficiales de HL7 Australia

### Paso 19 — Resultados de patología de `au-fhir-test-data`

HL7 Australia publica en GitHub recursos de prueba que siguen los perfiles AU Core. Sirven para comprobar que tu código entiende resultados australianos "de verdad" en FHIR, ya en LOINC SI y UCUM.

**Explicación línea por línea**

- `subprocess.run(["git", "clone", "--depth", "1", AU_REPO, str(au_dir)], check=True)`: clona solo la última versión (`--depth 1`). `check=True` lanza un error si `git` falla. Solo se clona si no existe ya.
- `git -C <carpeta> rev-parse HEAD`: obtiene el *commit* exacto que usaste; queda en el manifiesto. **Por qué:** el repositorio cambia; sin el commit no podrías reproducir este resultado.
- `(au_dir / "au-fhir-test-data-set" / "au-core").glob("Observation-*.json")`: solo leemos `Observation`. **No** se leen ni se publican los recursos `Patient`: algunos identificadores de prueba podrían coincidir con números reales.
- `coding = next((c for c in ... if c.get("system") == "http://loinc.org"), None)`: el primer código LOINC del recurso.
- `vq = r.get("valueQuantity") or {}`: el valor; si el resultado está **suprimido**, `valueQuantity` no trae `value` y en su lugar lleva la extensión `data-absent-reason` (por ejemplo `masked`).
- `ok_unit = vq.get("system") == "http://unitsofmeasure.org"`: la unidad debe venir en UCUM.
- `conf["value_si"] = [...]`: pasamos cada resultado por **tu** `to_si`; como ya están en SI, el valor no debe cambiar (`assert`).
- `au["motivo"] = np.select([...], [...], "ok")`: `np.select` evalúa las condiciones en orden y asigna la etiqueta de la primera que se cumple: `fuera_de_alcance` (otro LOINC, como presión arterial o el cociente colesterol/HDL `32309-7`), `sin_valor` (suprimido) u `ok`. Lo que no entra se cuenta y se muestra con su motivo; no se descarta en silencio.
- Observa los dos archivos de HDL con el mismo valor: uno es una **variante de prueba** (cambia solo el texto del rango de referencia). Los conjuntos de prueba oficiales incluyen variantes así a propósito, para que pruebes cómo tu sistema maneja casos límite.

In [ ]:
AU_REPO = "https://github.com/hl7au/au-fhir-test-data"
au_dir = EXT_DIR / "au-fhir-test-data"
if not au_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", AU_REPO, str(au_dir)], check=True)
au_commit = subprocess.run(["git", "-C", str(au_dir), "rev-parse", "HEAD"], check=True,
                           capture_output=True, text=True).stdout.strip()
print("au-fhir-test-data commit:", au_commit)

recs = []
for path in sorted((au_dir / "au-fhir-test-data-set" / "au-core").glob("Observation-*.json")):
    r = json.loads(path.read_text(encoding="utf-8"))
    coding = next((c for c in r.get("code", {}).get("coding", []) if c.get("system") == "http://loinc.org"), None)
    if coding is None:
        continue
    vq = r.get("valueQuantity") or {}
    absent = [e.get("valueCode") for e in vq.get("extension", [])
              if e.get("url") == "http://hl7.org/fhir/StructureDefinition/data-absent-reason"]
    recs.append({"file": path.name, "loinc_code": coding["code"], "effective": r.get("effectiveDateTime"),
                 "value": vq.get("value"), "unit": vq.get("code"), "ok_unit": vq.get("system") == "http://unitsofmeasure.org",
                 "absent_reason": absent[0] if absent else None})
au = pd.DataFrame(recs)
in_scope = au["loinc_code"].isin(LOINC_REF["loinc_code"])
conf = au[in_scope & au["value"].notna()].copy()
component = conf["loinc_code"].map(LOINC_REF.set_index("loinc_code")["component"])
conf["value_si"] = [to_si(c, v, u)[0] for c, v, u in zip(component, conf["value"], conf["unit"])]
assert conf["ok_unit"].all() and np.allclose(conf["value_si"], conf["value"].astype(float))
print(f"{len(conf)} resultados AU Core en alcance pasan la conformidad")
au["motivo"] = np.select([~in_scope, au["value"].isna()], ["fuera_de_alcance", "sin_valor"], "ok")
print(au["motivo"].value_counts().to_dict())
au.loc[in_scope, ["file", "loinc_code", "value", "unit", "absent_reason", "motivo"]]

## Parte G — Unir HCE y laboratorio sin contar dos veces

### Paso 20 — Fuente primaria y conciliación de ida y vuelta

El mismo análisis puede llegar por la HCE y por el laboratorio. Si los sumas, duplicas resultados; si eliges al azar, pierdes trazabilidad. La regla habitual es que **el sistema que produce el dato (el laboratorio) tiene precedencia**.

**Explicación línea por línea**

- `MATCH_KEYS = ["patient_id", "analyte", "collected_ts"]`: mismo paciente, mismo analito y misma hora de toma. Se compara por **analito**, no por código: la HCE trae `2093-3` (mg/dL) y el laboratorio `14647-2` (mmol/L). Aquí la hora coincide exactamente porque ambos describen la misma muestra; en producción se usa una ventana de tiempo y se revisan los casos dudosos.
- `lab["is_primary"] = lab["value_si"].notna()`: solo un valor utilizable puede ser primario.
- HbA1c NGSP del laboratorio (`4548-4`) → no primaria: para cada muestra se elige **una** cifra, la IFCC en mmol/mol (SI).
- `feed_keys` y `hit`: marca las filas de la HCE que tienen un resultado primario del laboratorio para la misma clave; `superseded_by_feed=True` y dejan de ser primarias, **pero no se borran**.
- **Conciliación de ida y vuelta:** une cada resultado del laboratorio con su par de la HCE y compara `value_si`. La tolerancia suma tres términos: el redondeo del laboratorio (media unidad del último decimal informado, convertida a SI con `value_si / value_num_source`), un 0,2 % por las pequeñas diferencias entre factores publicados (0,02586 frente a 10/386,664) y, solo para HbA1c, 0,05 mmol/mol, porque la fórmula de la Australian Diabetes Society y la ecuación maestra del NGSP difieren ligeramente (menos de 0,03 mmol/mol entre 2 % y 20 %). Se excluyen los censurados (`<30` no es un valor exacto) y las correcciones (la HCE tiene el valor anterior).
- `assert (~rt["ok"]).sum() == 0`: si una conversión estuviera mal (por ejemplo, un factor de glucosa equivocado), cientos de pares fallarían aquí.

In [ ]:
MATCH_KEYS = ["patient_id", "analyte", "collected_ts"]


def mark_primary(lab):
    lab = lab.copy()
    lab["is_primary"] = lab["value_si"].notna()
    lab.loc[(lab["source"] == "pathlab_feed") & (lab["loinc_code"] == "4548-4"), "is_primary"] = False
    feed_keys = lab.loc[(lab["source"] == "pathlab_feed") & lab["is_primary"], MATCH_KEYS].drop_duplicates()
    hit = lab[MATCH_KEYS].merge(feed_keys.assign(_hit=1), on=MATCH_KEYS, how="left")["_hit"].notna().to_numpy()
    superseded = (lab["source"] == "synthea_ehr").to_numpy() & hit
    lab["superseded_by_feed"] = superseded
    lab.loc[superseded, "is_primary"] = False
    return lab, int(superseded.sum())


lab, n_superseded = mark_primary(lab)

pairs = lab[lab["source"] == "pathlab_feed"].merge(
    lab.loc[lab["source"] == "synthea_ehr", MATCH_KEYS + ["value_si"]], on=MATCH_KEYS, suffixes=("", "_ehr"))
rt = pairs[(pairs["comparator"] == "=") & (pairs["status"] == "final")].copy()
dec = rt["source_value_text"].map(lambda t: len(t.split(".")[1]) if "." in t else 0)
rt["tol"] = (0.5 * 10.0 ** (-dec) * (rt["value_si"] / rt["value_num_source"]).abs()
             + 0.002 * rt["value_si"].abs() + np.where(rt["analyte"] == "hba1c", 0.05, 0.0))
rt["abs_diff"] = (rt["value_si"] - rt["value_si_ehr"]).abs()
rt = rt.sort_values("abs_diff").drop_duplicates(["source_record_id", "loinc_code"])
rt["ok"] = rt["abs_diff"] <= rt["tol"]
corrected = pairs[pairs["status"] == "corrected"]
print(f"HCE reemplazadas por el laboratorio: {n_superseded:,}")
print(f"Pares conciliados: {len(rt):,} | fuera de tolerancia: {(~rt['ok']).sum()} | correcciones que la HCE no tiene: {len(corrected)}")
assert (~rt["ok"]).sum() == 0
rt.groupby("analyte")["abs_diff"].max()

### Paso 21 — Guardar `silver.lab_results` con su manifiesto de linaje

**Explicación línea por línea**

- `lab["loinc_version"] = LOINC_VERSION`: cada fila sabe contra qué versión de LOINC se estandarizó.
- `lab.to_parquet(SILVER_DIR / "lab_results.parquet", index=False)`: la tabla queda junto al resto de silver; la semana 8 la cargará en OMOP.
- `def sha256_of(path):` huella del archivo leído por bloques de 1 MiB (el mismo patrón de la semana 1).
- `manifest = {...}`: entradas (extracto, vinculación, observaciones de silver, *crosswalk*, commit de `au-fhir-test-data`) con su huella, y conteos de salida. **Con esto puedes responder a una auditoría:** "¿con qué versión del mapeo se calculó este colesterol?".

In [ ]:
lab["loinc_version"] = LOINC_VERSION
lab.to_parquet(SILVER_DIR / "lab_results.parquet", index=False)


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


manifest = {
    "inputs": {name: sha256_of(path) for name, path in {
        "lab_feed_extract.csv": RAW_DIR / "lab_feed_extract.csv", "lab_ur_xref.csv": RAW_DIR / "lab_ur_xref.csv",
        "silver/observations.parquet": SILVER_DIR / "observations.parquet",
        "loinc_crosswalk.csv": REF_DIR / "loinc_crosswalk.csv"}.items()},
    "au_fhir_test_data_commit": au_commit,
    "loinc_version": LOINC_VERSION,
    "rows": {"lab_results": len(lab), "primary": int(lab["is_primary"].sum()),
             "superseded_by_feed": n_superseded, "quarantine": len(quarantine)},
}
(SILVER_DIR / "lab_results_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest["rows"], indent=2))
lab.groupby(["source", "is_primary"]).size()

## Parte H — Auditoría de unidades en la HCE

### Paso 22 — ¿Cada código usa una sola unidad? ¿Los valores son físicamente posibles?

**Explicación línea por línea**

- `units = obs[obs["value_num"].notna()].groupby("loinc_code").agg(...)`: por código LOINC, cuántas unidades distintas aparecen, cuáles son y la mediana del valor.
- `units[units["n_units"] > 1]`: un mismo LOINC con dos unidades es una señal de alarma. En Synthea aparece `33914-3` (filtrado glomerular estimado) con `mL/min` y `mL/min/{1.73_m2}`: no son la misma magnitud (la segunda está normalizada por superficie corporal), así que no deben mezclarse.
- Varias enzimas (`1742-6`, `1920-8`, `6768-6`) mezclan `U/L` y `[iU]/L`. En la práctica clínica se usan como sinónimos, pero en UCUM son unidades distintas (`U` se define como µmol/min; `[iU]` es una unidad arbitraria) y no se convierten automáticamente: equipararlas es una decisión que se documenta. `10834-0` (globulina) mezcla `g/L` y `g/dL`, que sí se convierten (factor 10), pero primero hay que confirmar cuál es la unidad verdadera de cada fila.
- `units.loc[["2885-2"]]`: proteína total con unidad `g/dL` y mediana ≈ 67. Eso serían unos 670 g/L: más de la mitad de la masa de un litro de plasma (≈ 1.025 g) sería proteína, algo físicamente imposible. El patrón numérico coincide con **g/L**. **Lección práctica:** una unidad declarada no es garantía; el perfil de valores la confirma o la desmiente. Lo correcto es reportarlo a la fuente, no "corregirlo" en silencio.
- `units_report.to_csv(...)`: el hallazgo queda documentado.

In [ ]:
units = (obs[obs["value_num"].notna()]
         .groupby("loinc_code")
         .agg(description=("description", "first"), n_units=("unit", "nunique"),
              units=("unit", lambda s: sorted(s.dropna().unique())), median=("value_num", "median"), n=("value_num", "size")))
units_report = pd.concat([units[units["n_units"] > 1], units.loc[["2885-2"]]])
units_report.to_csv(REPORTS_DIR / "ehr_unit_audit.csv")
units_report

## Parte I — Del *crosswalk* a recursos FHIR de terminología (temas del curso ADHA)

Tu *crosswalk* es una tabla. En FHIR, la misma información tiene recursos propios: un `CodeSystem` para los códigos locales del laboratorio y un `ConceptMap` para su relación con LOINC. Así otros sistemas pueden consumir el mapeo con operaciones estándar como `$translate`.

**Estrategias de codificación (curso, tema 4), en términos de esta semana:**

- **Códigos estándar nativos:** la HCE ya trae LOINC. Es lo más simple de consumir, pero depende de que el emisor codifique bien.
- **Códigos locales con mapeo:** el extracto del laboratorio. Conserva el código original y añade el estándar; exige mantener y versionar el mapeo, que es lo que hiciste.
- **ValueSets híbridos:** códigos estándar más algunos locales en el mismo conjunto. Ayudan en la transición, pero lo local no es interoperable fuera de tu organización.

### Paso 23 — Un `CodeSystem` para los códigos locales

**Explicación línea por línea**

- `LOCAL_CS_URL = "http://example.org/fhir/CodeSystem/pathlab-sim-tests"`: todo sistema de códigos necesita una URL canónica **permanente**. `example.org` es un dominio reservado para ejemplos; un laboratorio real usaría un dominio propio.
- `"version": "1.0.0"`: la versión del catálogo del laboratorio. Si el laboratorio cambia el significado de un código, debe publicar una versión nueva; nunca reutilizar el código.
- `"status": "active"`, `"experimental": True`: está en uso, pero es material de práctica.
- `"caseSensitive": True`: `CHOL` y `chol` serían códigos distintos. Por eso normalizaste mayúsculas y espacios en el Paso 15.
- `"content": "complete"`: el recurso contiene **todos** los conceptos del sistema.
- `"concept": [...]`: un concepto por código local, con su nombre.

In [ ]:
LOCAL_CS_URL = "http://example.org/fhir/CodeSystem/pathlab-sim-tests"
local_codes = crosswalk.drop_duplicates("local_code").sort_values("local_code")
code_system = {
    "resourceType": "CodeSystem", "url": LOCAL_CS_URL, "version": "1.0.0", "name": "PathlabSimTests",
    "status": "active", "experimental": True, "caseSensitive": True, "content": "complete",
    "concept": [{"code": r.local_code, "display": r.local_name} for r in local_codes.itertuples(index=False)],
}
(REF_DIR / "pathlab_codesystem.json").write_text(json.dumps(code_system, indent=2), encoding="utf-8")
print(len(code_system["concept"]), "conceptos locales")

### Paso 24 — El `ConceptMap`: equivalencia y dependencia de la unidad

**Explicación línea por línea**

- `group.source` y `group.target`: de qué sistema a qué sistema se mapea; `targetVersion` registra la versión de LOINC del mapeo.
- `element`: un elemento por código local; `target`: uno por cada LOINC posible de ese código.
- `"equivalence"`: el grado de equivalencia de FHIR R4. Se toma de la columna `equivalence` del *crosswalk* (Paso 10): `equivalent` para lo aprobado y `unmatched` para lo que no tiene destino. Otros valores posibles son `wider` (el destino es más amplio y se pierde detalle), `narrower` (el destino es más específico), `inexact` o `relatedto`; si un mapeo es `narrower` o `inexact`, la especificación exige un comentario.
- `"dependsOn": [{"property": UNIT_PROPERTY, "system": UCUM, "value": unit}]`: el destino **depende** de otro dato, la unidad del resultado. Así se expresa en FHIR que `CHOL` en mmol/L es `14647-2` y en mg/dL es `2093-3`. `UNIT_PROPERTY` apunta al elemento que contiene la unidad (`Quantity.code`).
- `"comment"`: la nota del revisor viaja con el mapeo.
- `"version": crosswalk_sha[:12]`: la versión del `ConceptMap` es la huella del *crosswalk* del que salió. Si cambia una sola fila, cambia la versión.

In [ ]:
UCUM = "http://unitsofmeasure.org"
UNIT_PROPERTY = "http://hl7.org/fhir/StructureDefinition/Quantity#Quantity.code"
crosswalk_sha = hashlib.sha256((REF_DIR / "loinc_crosswalk.csv").read_bytes()).hexdigest()

elements = []
for local_code, rows in crosswalk.groupby("local_code", sort=True):
    targets = []
    for r in rows.itertuples(index=False):
        t = {"equivalence": r.equivalence,
             "dependsOn": [{"property": UNIT_PROPERTY, "system": UCUM, "value": r.units}]}
        if r.equivalence != "unmatched":
            t.update({"code": r.loinc_code, "display": r.long_common_name})
        t["comment"] = r.review_note if isinstance(r.review_note, str) else "Pendiente de revisión terminológica"
        targets.append(t)
    elements.append({"code": local_code, "display": rows["local_name"].iloc[0], "target": targets})

concept_map = {
    "resourceType": "ConceptMap", "url": "http://example.org/fhir/ConceptMap/pathlab-sim-to-loinc",
    "version": crosswalk_sha[:12], "name": "PathlabSimToLoinc", "status": "active", "experimental": True,
    "group": [{"source": LOCAL_CS_URL, "sourceVersion": code_system["version"],
               "target": "http://loinc.org", "targetVersion": LOINC_VERSION, "element": elements}],
}
(REF_DIR / "pathlab_to_loinc_conceptmap.json").write_text(json.dumps(concept_map, indent=2), encoding="utf-8")
next(e for e in elements if e["code"] == "CHOL")

### Paso 25 — `$translate` sobre tu `ConceptMap` y prueba de coherencia

`ConceptMap/$translate` recibe un código y devuelve `result` (verdadero solo si al menos una coincidencia tiene una equivalencia distinta de `unmatched` o `disjoint`) y una lista de `match`, cada uno con su equivalencia y su concepto. Aquí implementas esa lógica localmente para probar tu `ConceptMap` antes de publicarlo en un servidor.

**Explicación línea por línea**

- `def translate(cm, system, code, unit):` recorre los grupos cuyo `source` es el sistema pedido y los elementos con ese código.
- `if deps and (UNIT_PROPERTY, unit) not in deps: continue`: si el destino depende de una unidad, solo cuenta cuando la unidad coincide. Es el equivalente local del parámetro `dependency` de `$translate`.
- `result = any(m["equivalence"] not in ("unmatched", "disjoint") ...)`: la regla de la especificación.
- **Prueba de coherencia:** para cada par código + unidad del extracto limpio, `$translate` sobre el `ConceptMap` debe dar el mismo LOINC que el *crosswalk* aplicado en el Paso 16. Dos representaciones del mismo mapeo deben responder igual.
- `LPA` debe dar `result: False` con equivalencia `unmatched`.

In [ ]:
def translate(cm, system, code, unit):
    matches = []
    for g in cm["group"]:
        if g["source"] != system:
            continue
        for el in g["element"]:
            if el["code"] != code:
                continue
            for t in el.get("target", []):
                deps = {(d["property"], d["value"]) for d in t.get("dependsOn", [])}
                if deps and (UNIT_PROPERTY, unit) not in deps:
                    continue
                concept = {"system": g["target"], "code": t["code"]} if "code" in t else None
                matches.append({"equivalence": t["equivalence"], "concept": concept})
    result = any(m["equivalence"] not in ("unmatched", "disjoint") for m in matches)
    return {"result": result, "match": matches}


pairs = clean[["local_code", "units"]].drop_duplicates()
expected = crosswalk.loc[crosswalk["status"] == "approved"].set_index(["local_code", "units"])["loinc_code"]
mismatches = 0
for r in pairs.itertuples(index=False):
    out = translate(concept_map, LOCAL_CS_URL, r.local_code, r.units)
    got = next((m["concept"]["code"] for m in out["match"] if m["concept"]), None)
    if got != expected.get((r.local_code, r.units)):
        mismatches += 1
print("Pares código+unidad:", len(pairs), "| discrepancias ConceptMap vs crosswalk:", mismatches)
assert mismatches == 0
translate(concept_map, LOCAL_CS_URL, "LPA", "nmol/L")

### Paso 26 — Un resultado de laboratorio como `Observation` FHIR

**Explicación línea por línea**

- `sample = mapped[(mapped["local_code"] == "CREAT") & (mapped["comparator"] == "<") & mapped["patient_id"].notna()].iloc[0]`: un resultado censurado real del extracto.
- `"code": {"coding": [local, loinc]}`: el código original del laboratorio **y** el LOINC en el mismo `CodeableConcept`. Quien recibe puede usar el estándar sin perder la trazabilidad al código de origen.
- `"category"` con `laboratory`: la categoría de observación del vocabulario de HL7.
- `"valueQuantity": {"comparator": "<", "value": 30, "system": UCUM, "code": "umol/L"}`: el `<30` en FHIR; `float(...)` convierte el número de NumPy en un número de Python, el tipo que espera JSON. `system` + `code` identifican la unidad en UCUM; `unit` es el texto legible.
- `"effectiveDateTime": sample["collected_at"]`: la hora con su desfase (`+10:00`), tal como la envió el laboratorio.
- `loinc_in_obs = [c["code"] for c in ... if c["system"] == "http://loinc.org"]`: elegir el código por sistema, igual que en las vistas de la semana 5.
- En Australia, AU Base asocia el código de un resultado de patología (perfil *AU Base Pathology Result*) a los conjuntos SPIA del RCPA, los mismos que consultaste en el Paso 12.

In [ ]:
sample = mapped[(mapped["local_code"] == "CREAT") & (mapped["comparator"] == "<") & mapped["patient_id"].notna()].iloc[0]
obs_fhir = {
    "resourceType": "Observation", "id": "lab-demo-1",
    "status": {"F": "final", "C": "corrected"}[sample["status"]],
    "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category",
                              "code": "laboratory"}]}],
    "code": {"coding": [{"system": LOCAL_CS_URL, "code": sample["local_code"], "display": sample["local_name"]},
                        {"system": "http://loinc.org", "code": sample["loinc_code"],
                         "display": LOINC_REF.set_index("loinc_code").loc[sample["loinc_code"], "long_common_name"]}],
             "text": sample["local_name"]},
    "subject": {"reference": f"Patient/{sample['patient_id']}"},
    "effectiveDateTime": sample["collected_at"],
    "valueQuantity": {"comparator": sample["comparator"], "value": float(sample["value_num"]), "unit": "µmol/L",
                      "system": UCUM, "code": sample["units"]},
}
loinc_in_obs = [c["code"] for c in obs_fhir["code"]["coding"] if c["system"] == "http://loinc.org"]
print("LOINC elegido por sistema:", loinc_in_obs)
obs_fhir

## Cierre

**Qué construiste:** un flujo completo de estandarización de laboratorio, como el que se hace al integrar un laboratorio nuevo en un hospital o en un repositorio de investigación: perfil por volumen → ejes LOINC → mapeo semiautomático con revisión humana y versión → validación contra el servidor nacional y SPIA → armonización SI con UCUM → cuarentena → precedencia entre fuentes y conciliación numérica.

**Preguntas para tu bitácora**

1. ¿Por qué `CHOL` en mmol/L y `CHOL` en mg/dL deben mapearse a códigos LOINC distintos?
2. ¿Qué pasaría con los promedios de creatinina si guardaras `<30` como 30? ¿Y si lo descartaras?
3. ¿Por qué no se puede aceptar automáticamente un mapeo inferido desde la unidad "%"?
4. La glucosa en sangre total y en plasma no tienen el mismo valor para una misma persona. ¿Qué eje de LOINC lo distingue y qué riesgo hay en unir ambas como el mismo analito?
5. Busca en el buscador de LOINC un código candidato para `LPA` (lipoproteína(a) en nmol/L) y razona su propiedad y su sistema antes de aprobarlo.
6. ¿Qué ganas al publicar el mapeo como `ConceptMap` en vez de un CSV, y qué se rompería si omitieras `dependsOn`?

**Commit sugerido:** `feat(s07): mapeo LOINC con revisión, validación NCTS/SPIA y armonización SI de laboratorio`

**Qué sube al repo:** el notebook y `loinc_crosswalk.csv` (metadato de mapeo, sin datos de pacientes). **Qué no sube:** nada de `data/` (el extracto, la vinculación y `lab_results.parquet` contienen identificadores de pacientes, aunque sean sintéticos).